

# GoC Spending Analysis (2017-2026)

Since the mid-2000s, contracts larger than $10,000 have been published on each department’s Proactive Disclosure websites.

But since 2016, departments have begun publishing contracting data on the Open Government website in a government-wide CSV dataset.


The objective of this framework is to move beyond "point-in-time" reporting. By applying advanced data transformation techniques, this project transforms raw, messy CSV/Excel outputs into a cleansed, normalized, and temporally distributed dataset.


In [1]:
# Impoprting imnportant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import time

# Data Source

The foundational data for this analysis is a bulk export of federal contract records, ingested as contract_data.xlsx. This dataset consists of 1,048,575 individual records spanning 43 distinct variables.

This dataset  typically list:

* the name of the vendor
* a reference number
* procurement_id (unique identifier for each contract)
* the contract date (when it was recorded in the department’s financial system)
* the contract period (start and end), or a delivery date
the total value of the contract
* a description of what work was requested

and many more columns.


In [3]:
df_contracts = pd.read_excel('data/contract_data.xlsx')

In [4]:
df_contracts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 43 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   reference_number                    1048574 non-null  object 
 1   procurement_id                      1035208 non-null  object 
 2   vendor_name                         1048307 non-null  object 
 3   vendor_postal_code                  266634 non-null   object 
 4   buyer_name                          240823 non-null   object 
 5   contract_date                       1048575 non-null  object 
 6   economic_object_code                798678 non-null   object 
 7   description_en                      1046937 non-null  object 
 8   description_fr                      1046554 non-null  object 
 9   contract_period_start               849108 non-null   object 
 10  delivery_date                       1025064 non-null  object 
 11  contract_va

# Data Preparation

This dataset includes significant amount of noisy data. . To maintain a responsive environment for appropriate context we immediately dropped 24 non-essential columns.

* Fields like **agreement_type_code** or **standing_offer_number** explain the mechanism of the contract but don't change the value of the spend.

* We dropped **description_fr** and **comments_fr** to save memory. Since our vendor normalization logic is optimized for English string matching, keeping the French duplicates would double our memory usage without adding unique data points.

* Columns like **agreement_type_code, standing_offer_number,** and **instrument_type** were removed as they provide context on how a contract was awarded but do not impact the calculation of how much was spent.

* All other columns were removed too since they have no real affect on our anlaysis.

In [114]:
columns_to_drop = [
    'buyer_name',
    'economic_object_code',
    'description_fr',
    'comments_fr',
    'trade_agreement',
    'land_claims',
    'intellectual_property',
    'potential_commercial_exploitation',
    'former_public_servant',
    'additional_comments_en',
    'additional_comments_fr',
    'agreement_type_code',
    'commodity_code',
    'limited_tendering_reason',
    'trade_agreement_exceptions',
    'indigenous_business',
    'contracting_entity',
    'standing_offer_number',
    'article_6_exceptions',
    'award_criteria',
    'socioeconomic_indicator',
    'potential_commercial_exploitation',
    'ministers_office',
    'instrument_type'
]

df = df_contracts.drop(columns=columns_to_drop, errors='ignore')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   reference_number                    1048574 non-null  object 
 1   procurement_id                      1035208 non-null  object 
 2   vendor_name                         1048307 non-null  object 
 3   vendor_postal_code                  266634 non-null   object 
 4   contract_date                       1048575 non-null  object 
 5   description_en                      1046937 non-null  object 
 6   contract_period_start               849108 non-null   object 
 7   delivery_date                       1025064 non-null  object 
 8   contract_value                      1048279 non-null  float64
 9   original_value                      763124 non-null   float64
 10  amendment_value                     643215 non-null   float64
 11  comments_en

We make the **comments_en and owner_org_title **columns to uppercase and strip whitespace. In public datasets, "National Defence" and "NATIONAL DEFENCE" are treated as two different departments by Python. By standardizing the case early, we ensure that when we group contracts by owner_org, we aren't missing amendments due to simple casing typos.

In [115]:
# Convert the 'comments_en' column to all uppercase to have all matching strings
df['comments_en'] = df['comments_en'].str.upper()

In [128]:
# Split by '|' and take the first element (index 0), then strip whitespace
df['owner_org_title'] = df['owner_org_title'].str.split('|').str[0].str.strip()

# Calculating start_year and end_year


## Calcluting start_year

To calculate this start_year we make use of year extracted from contract_period_year .

We also calculate contract_year from contract_date (date when contract was given)  for saftey reasons cause we cant have a scenario where **contract_year > start_year** since contract_year is year when contract was given and start_year is when the contract services started

In [116]:
import pandas as pd
import numpy as np

# Convert to datetime first
df['contract_date'] = pd.to_datetime(df['contract_date'], errors='coerce')
df['contract_period_start'] = pd.to_datetime(df['contract_period_start'], errors='coerce')

# Extracting the required years
df['contract_year'] = df['contract_date'].dt.year
df['start_year'] = df['contract_period_start'].dt.year

# Comparing the year and replacing where necessary
df['start_year'] = np.where(
    df['contract_year'] > df['start_year'],
    df['contract_year'],
    df['start_year']
)

# Fill the missing values with contract year
df['start_year'] = df['start_year'].fillna(df['contract_year'])

## Calculate the end_year

We calculate the end_year for information when the contract ends.

In this there were many anomolies observed during exploring the CSV. There were unreaslitic future values which arent possible. So we decided to keep end_year 2029 since contracts given in 2026 could potentially be till 2029 assuming a real world scenario.

If end_year was found to be greater then 2029. We fill it with NA and then replace those NAs with start year.



*   Note : This assumption was made based on real world scenarios and maybe be wrong .




In [118]:
# Slice deliver dates into string
df['end_year'] = df['delivery_date'].astype(str).str[0:4]

# Convert that text year into a proper number
df['end_year'] = pd.to_numeric(df['end_year'], errors='coerce').astype('Int64')

In [121]:
# Condition if greater than 2029 fill NA
condition = (df['end_year'] > 2029).fillna(False)

# Cap the future dates using our safe condition
df['end_year'] = np.where(
    condition,
    df['start_year'],
    df['end_year']
)

# If there are still NAs use start_year
df['end_year'] = df['end_year'].fillna(df['start_year'])

# Basic Error Detection

For both data sources, a set of basic error checks were applied. Contracts without a source(start)  year or without a contract value (or one of $0) were flagged, to avoid including them in subsequent analyses or exports. CSV entries that didn’t include a departmental owner acronym, a vendor name, or a contract value were also flagged separately.

In [123]:
# Transform reporting_period format ((e.g., '2019-2020-Q4' -> '201920-Q4')

def transform_reporting_period(period):
    if pd.isna(period):
        return period

    period = str(period).strip()

    match = re.match(r'^(\d{4})-(\d{4})-(Q\d)', period)
    if match:
        start_year = match.group(1)
        end_year_short = match.group(2)[-2:]
        quarter = match.group(3)
        return f"{start_year}{end_year_short}-{quarter}"

    return period

df['reporting_period'] = df['reporting_period'].apply(transform_reporting_period)


# Split the dataset into clean_df and error_df
# Critical_errors
critical_errors = (
    df['start_year'].isna() |
    df['contract_value'].isna() |
    (df['contract_value'] == 0) |
    df['procurement_id'].isna()
)

# clean_df (rows without critical_errors )
clean_df = df[~critical_errors].copy().reset_index(drop=True)

# error_df ( rows with critical_errors )
error_df = df[critical_errors].copy().reset_index(drop=True)

# Step 3: Quick stats
print(f"Total rows in raw dataset: {len(df)}")
print(f"Rows cleared for analysis (clean_df): {len(clean_df)}")
print(f"Rows with critical errors (error_df): {len(error_df)}")

Total rows in raw dataset: 1048575
Rows cleared for analysis (clean_df): 1021875
Rows with critical errors (error_df): 26700


In [126]:
clean_df

,reference_number,procurement_id,vendor_name,vendor_postal_code,contract_date,description_en,contract_period_start,delivery_date,contract_value,original_value,...,country_of_vendor,solicitation_procedure,indigenous_business_excluding_psib,number_of_bids,reporting_period,owner_org,owner_org_title,contract_year,start_year,end_year
0,C-2019-2020-Q4-1,P2000002,Simzer Design Inc.,NaN,2020-02-26,Communications professional services not elsew...,2019-11-15,2020-05-30 00:00:00,38900.25,15255.00,...,CA,TN,N,NaN,201920-Q4,casdo-ocena,Accessibility Standards Canada | Normes d’acce...,2020,2020.0,2020.0
1,C-2019-2020-Q4-2,P2000004,Breckenhill Inc.,NaN,2020-02-26,Other professional services not elsewhere spec...,2019-12-20,2020-06-30 00:00:00,39832.51,24789.38,...,CA,TN,N,NaN,201920-Q4,casdo-ocena,Accessibility Standards Canada | Normes d’acce...,2020,2020.0,2020.0
2,C-2019-2020-Q4-3,P2000009,Spinal Cord Injury Canada,NaN,2020-03-11,Translation services,2020-03-16,2020-07-31 00:00:00,38420.00,38420.00,...,CA,TN,N,NaN,201920-Q4,casdo-ocena,Accessibility Standards Canada | Normes d’acce...,2020,2020.0,2020.0
3,C-2019-2020-Q4-4,P2000011,CSA Group,NaN,2020-03-11,Management consulting,2020-03-02,2020-06-30 00:00:00,39550.00,39550.00,...,CA,TN,N,NaN,201920-Q4,casdo-ocena,Accessibility Standards Canada | Normes d’acce...,2020,2020.0,2020.0
4,C-2019-2020-Q4-5,P2000014,Simzer Design Inc.,NaN,2020-03-20,Communications professional services not elsew...,2020-03-02,2020-08-31 00:00:00,23571.80,23571.80,...,CA,TN,N,NaN,201920-Q4,casdo-ocena,Accessibility Standards Canada | Normes d’acce...,2020,2020.0,2020.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021870,C-2015-2016-Q3-00376,45366918,Nortrax Canada Inc,NaN,2015-11-27,Road motor vehicles,2015-11-27,2016-03-31 00:00:00,252360.15,252360.15,...,NaN,NaN,NaN,NaN,201516-Q3,pc,Parks Canada | Parcs Canada,2015,2015.0,2016.0
1021871,C-2015-2016-Q3-00377,45366371,General Motors of Canada Ltd.,NaN,2015-12-15,Road motor vehicles,2015-12-15,2016-03-31 00:00:00,22475.25,22475.25,...,NaN,NaN,NaN,NaN,201516-Q3,pc,Parks Canada | Parcs Canada,2015,2015.0,2016.0
1021872,C-2015-2016-Q3-00378,2015063100,FCA Canada Inc.,NaN,2015-11-13,Road motor vehicles,2015-11-13,2016-03-31 00:00:00,22521.45,22521.45,...,NaN,NaN,NaN,NaN,201516-Q3,pc,Parks Canada | Parcs Canada,2015,2015.0,2016.0
1021873,C-2015-2016-Q3-00379,2015059000,General Motors of Canada Ltd.,NaN,2015-10-29,Road motor vehicles,2015-10-29,2016-04-03 00:00:00,25543.35,25543.35,...,NaN,NaN,NaN,NaN,201516-Q3,pc,Parks Canada | Parcs Canada,2015,2015.0,2016.0


# Vendor Normalization

The contracting data only provideS  a free-text vendor name field, without a business number or other official identifier. Matching together data from the same vendor was challenging as a result. Variations in vendor names could result from:

* Legal Suffix Inconsistency: Variations like "Ltd.", "Inc.", "Corp", or "Ulc" being included or omitted.
* Regional Tagging: Inclusion of office locations (e.g., "Microsoft Canada" vs. "Microsoft Toronto").
* Linguistic Variations: Bilingual naming (e.g., "General Motors of Canada" vs. "General Motors du Canada").
* Structural Changes: Mergers, acquisitions, and the use of Joint Ventures (JVs) or resellers.




The first layer of normalization utilizes a "Master Mapping" approach. We ingest a curated vendor_comparison_master.csv

This matching was done by hand (looking up company names with similar keywords or character sequences in the database), it has a number of important limitations.


In [129]:
vendor_df = pd.read_csv('data/vendor_comparison_master.csv')
vendor_df

,vendor_name,vendor_name_clean,master_name
0,Simzer Design Inc.,SIMZER DESIGN,SIMZER DESIGN
1,Breckenhill Inc.,BRECKENHILL,BRECKENHILL
2,Spinal Cord Injury Canada,SPINAL CORD INJURY CANADA,SPINAL CORD INJURY
3,CSA Group,CSA GROUP,CSA GROUP
4,Simzer Design Inc.,SIMZER DESIGN,SIMZER DESIGN
...,...,...,...
1048570,Nortrax Canada Inc,NORTRAX CANADA,NORTRAX
1048571,General Motors of Canada Ltd.,GENERAL MOTORS OF CANADA,GENERAL MOTORS
1048572,FCA Canada Inc.,FCA CANADA,FCA
1048573,General Motors of Canada Ltd.,GENERAL MOTORS OF CANADA,GENERAL MOTORS


In [130]:
# Regex pattern to matche a single letter, a space, and another single letter at the start
pattern = r'^([A-Z]) ([A-Z])\b'

vendor_df['master_name'] = vendor_df['master_name'].apply(lambda x: re.sub(pattern, r'\1\2', str(x)))

print(vendor_df.head())

                 vendor_name          vendor_name_clean         master_name
0         Simzer Design Inc.              SIMZER DESIGN       SIMZER DESIGN
1           Breckenhill Inc.                BRECKENHILL         BRECKENHILL
2  Spinal Cord Injury Canada  SPINAL CORD INJURY CANADA  SPINAL CORD INJURY
3                  CSA Group                  CSA GROUP           CSA GROUP
4         Simzer Design Inc.              SIMZER DESIGN       SIMZER DESIGN


Character-Level Trigram Vectorization

Now we use a TfidfVectorizer with a char_wb analyzer and an ngram_range of (3,3).
* Unlike word-based matching, character trigrams (breaking "CANADA" into "CAN", "ANA", "NAD", "ADA") allow the system to recognize similarities despite typos, missing spaces, or truncated legal suffixes.
* To maintain performance on a dataset of this size, we implement Blocking Logic. Vendors are only compared to one another if they share the first 5 characters of their cleaned name. Within these blocks, we calculate a similarity matrix where:
Similarity Score≥0.78
* Setting the threshold at 0.78 balances the need to merge variations (like "ADVANCE" and "ADVANCED") while preventing the incorrect merging of distinct companies that share common roots.


In [131]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Ensure master_name is a string
vendor_df['master_name'] = vendor_df['master_name'].astype(str)

# 2. Count Frequencies to establish the "Canonical" (most popular) names
print("Calculating frequencies...")
name_counts = vendor_df['master_name'].value_counts()
unique_df = pd.DataFrame({'name': name_counts.index, 'freq': name_counts.values})

# 3. Create a temporary "Fluff-Free" column just for scoring
fluff_words = r'\b(TECHNOLOGIES|TECHNOLOGY|SOLUTIONS|SOLUTION|GROUP|ENTERPRISES|ENTERPRISE|CONSULTING|SYSTEMS|MANAGEMENT|ASSOCIATES|INTERNATIONAL)\b'

unique_df['clean_score_name'] = unique_df['name'].apply(
    lambda x: re.sub(fluff_words, '', str(x)).strip()
)
# Clean up any double spaces left behind
unique_df['clean_score_name'] = unique_df['clean_score_name'].apply(lambda x: re.sub(r'\s+', ' ', x))

# 4. Create the "Block" using the first 5 characters (instead of the whole first word)
unique_df['block'] = unique_df['clean_score_name'].str[:5]

# 5. Prepare for Fuzzy Matching
name_mapping = {}
# We can safely lower the threshold to 0.78 now that the fluff words are gone
threshold = 0.78
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 3))

print("Finding advanced fuzzy matches (this may take a minute)...")
grouped = unique_df.groupby('block')

for block, group_original in grouped:
    # Filter the group to include only entries that have a processable clean_score_name
    group = group_original[group_original['clean_score_name'].str.len() >= 3].copy()

    if len(group) < 2:
        continue

    # Get the original names and the stripped-down names from the filtered group
    names = group['name'].tolist()
    clean_names = group['clean_score_name'].tolist()

    # We measure similarity on the 'clean_names' (without the fluff words)
    tfidf_matrix = vectorizer.fit_transform(clean_names)
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    for i in range(len(names)):
        if names[i] in name_mapping:
            continue

        for j in range(i + 1, len(names)):
            if names[j] in name_mapping:
                continue

            if cosine_sim[i, j] >= threshold:
                # Map the less frequent name to the more frequent canonical name
                freq_i = unique_df[unique_df['name'] == names[i]]['freq'].iloc[0]
                freq_j = unique_df[unique_df['name'] == names[j]]['freq'].iloc[0]

                if freq_i >= freq_j:
                    name_mapping[names[j]] = names[i]
                else:
                    name_mapping[names[i]] = names[j]

# 6. Apply the mapping back to your original DataFrame
print(f"\nSuccess! Found {len(name_mapping)} advanced variations to merge.")

vendor_df['master_name'] = vendor_df['master_name'].map(lambda x: name_mapping.get(x, x))



Calculating frequencies...
Finding advanced fuzzy matches (this may take a minute)...

Success! Found 10445 advanced variations to merge.
Applying cleaning to the main dataset...


Despite our effort, several inherent limitations remain:
* **Arbitrary Joint Venture Assignment:** Joint ventures are typically associated with only one participating company based on whichever manual lookup or fuzzy match occurs first in the iteration.
* **Reseller Ambiguity:** Large software resellers are sometimes merged with the original manufacturer and sometimes treated as standalone entities, depending on how the procurement_id links the records.
* **Subsidiary Blurring:** Companies with vast holdings (e.g., telecommunications giants that also provide IT services) are merged into a single parent name. This makes per-industry spending analysis difficult, as distinct services are aggregated under one corporate umbrella.
* **Historical M&A Inconsistency:** While the model attempts to normalize companies that have merged, this is limited to information discoverable through the dataset or cursory external research, leading to potential inconsistencies in older records.


In [132]:
vendor_df

,vendor_name,vendor_name_clean,master_name
0,Simzer Design Inc.,SIMZER DESIGN,SIMZER DESIGN
1,Breckenhill Inc.,BRECKENHILL,BRECKENHILL
2,Spinal Cord Injury Canada,SPINAL CORD INJURY CANADA,SPINAL CORD INJURY
3,CSA Group,CSA GROUP,CSA GROUP
4,Simzer Design Inc.,SIMZER DESIGN,SIMZER DESIGN
...,...,...,...
1048570,Nortrax Canada Inc,NORTRAX CANADA,NORTRAX
1048571,General Motors of Canada Ltd.,GENERAL MOTORS OF CANADA,GENERAL MOTORS
1048572,FCA Canada Inc.,FCA CANADA,FCA
1048573,General Motors of Canada Ltd.,GENERAL MOTORS OF CANADA,GENERAL MOTORS


In [133]:
# Map the master_name with vendor_name column in the original dataframe (clean_df)
mapping_series = vendor_df.drop_duplicates('vendor_name').set_index('vendor_name')['master_name']
clean_df['master_name'] = clean_df['vendor_name'].map(mapping_series)

In [135]:
# Drop the oroginal vendor_name column after normalization
clean_df.drop(columns=['vendor_name'], inplace=True)

# Date validation

Here we clean the reporting_period column cause it has many null values as well as discrpaencies and typos which would be fatal in our analysis if carried forward.

In [138]:
# Function to clean reporting_period column
def standardize_quarter(row):
    # Looking specifically at the 'reporting_period' column
    val = str(row.get('reporting_period', '')).strip().upper()

    # Fixing obvious typos
    val = val.replace('20015', '2015')
    val = val.replace('2108', '2018')
    val = val.replace('201818', '201819')

    # Extract the first valid 4-digit year and the Quarter
    year_match = re.search(r'(20\d{2})', val)
    q_match = re.search(r'(Q[1-4])', val)

    # If both exist, enforce the YYYYYY-QX format
    if year_match and q_match and len(val) >= 5:
        start_year = int(year_match.group(1))

        # Last two digits of the year (e.g., 2019 + 1 = 2020 -> "20")
        next_year_suffix = str(start_year + 1)[-2:]
        q_val = q_match.group(1)

        return f"{start_year}{next_year_suffix}-{q_val}"


    # fallback logic : The most accurate date to determine the reporting period is the contract_date
    if pd.notnull(row.get('contract_date')):


        c_date = pd.to_datetime(row['contract_date'], errors='coerce')

        if pd.notnull(c_date):
            month = c_date.month
            year = c_date.year

            # Derived based on Government of Canada fiscal year
            if 4 <= month <= 6:
                q = 'Q1'
                fy_start = year
            elif 7 <= month <= 9:
                q = 'Q2'
                fy_start = year
            elif 10 <= month <= 12:
                q = 'Q3'
                fy_start = year
            else:
                # January through March fall into Q4 of the previous year
                q = 'Q4'
                fy_start = year - 1

            # Last two digits of the next year (e.g., 2019 + 1 = 2020 -> "20")
            fy_end_suffix = str(fy_start + 1)[-2:]

            # Format : YYYYYY-Q
            return f"{fy_start}{fy_end_suffix}-{q}"


    return np.nan



clean_df['clean_reporting_period'] = clean_df.apply(standardize_quarter, axis=1)


print(clean_df[['reporting_period', 'clean_reporting_period', 'delivery_date']].drop_duplicates().head(20))


   reporting_period clean_reporting_period        delivery_date
0         201920-Q4              201920-Q4  2020-05-30 00:00:00
1         201920-Q4              201920-Q4  2020-06-30 00:00:00
2         201920-Q4              201920-Q4  2020-07-31 00:00:00
4         201920-Q4              201920-Q4  2020-08-31 00:00:00
5         202021-Q1              202021-Q1  2020-12-31 00:00:00
6         202021-Q1              202021-Q1  2021-03-31 00:00:00
7         202021-Q1              202021-Q1  2020-11-30 00:00:00
8         202021-Q2              202021-Q2  2020-01-31 00:00:00
9         202021-Q3              202021-Q3  2021-02-28 00:00:00
10        202021-Q3              202021-Q3  2021-03-31 00:00:00
11        202021-Q3              202021-Q3  2021-12-15 00:00:00
12        202021-Q4              202021-Q4  2021-06-01 00:00:00
13        202021-Q4              202021-Q4  2021-09-30 00:00:00
15        202021-Q4              202021-Q4  2027-12-31 00:00:00
16        202021-Q4              202021-

In [139]:
clean_df['clean_reporting_period'].unique()

array(['201920-Q4', '202021-Q1', '202021-Q2', '202021-Q3', '202021-Q4',
       '202122-Q1', '202122-Q2', '202122-Q3', '202122-Q4', '202223-Q2',
       '202223-Q3', '202223-Q4', '202324-Q1', '202324-Q2', '202324-Q3',
       '202324-Q4', '202425-Q1', '202425-Q2', '202425-Q3', '202425-Q4',
       '202526-Q1', '202526-Q2', '202526-Q3', '201819-Q4', '201920-Q1',
       '201920-Q2', '201718-Q2', '201920-Q3', '201718-Q4', '201819-Q1',
       '201617-Q3', '201617-Q4', '201819-Q2', '201819-Q3', '201718-Q3',
       '201718-Q1', '202223-Q1', '201415-Q4', '201516-Q1', '201516-Q2',
       '201516-Q3', '201516-Q4', '201617-Q1', '201617-Q2', '201415-Q1',
       '201112-Q2', '201314-Q4', '201415-Q2', '201415-Q3', '200304-Q4',
       '200405-Q1', '200405-Q2', '200405-Q3', '200405-Q4', '200506-Q1',
       '200506-Q2', '200506-Q3', '200506-Q4', '200607-Q1', '200607-Q2',
       '200607-Q3', '200607-Q4', '200708-Q1', '200708-Q2', '200708-Q3',
       '200708-Q4', '200809-Q1', '200809-Q2', '200809-Q3', '2008

In [140]:
clean_df.drop(columns=['reporting_period'], inplace=True)

In [141]:
# Calculate effective_start_year and effective_end_year cause they might be different from contract_year and delivery_date
clean_df['effective_start_year'] = clean_df['start_year'].astype('Int64')
clean_df['effective_end_year'] = clean_df['end_year'].astype('Int64')


In [142]:
clean_df.drop(columns=['contract_period_start','delivery_date'], inplace=True)
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1021875 entries, 0 to 1021874
Data columns (total 23 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   reference_number                    1021874 non-null  object        
 1   procurement_id                      1021875 non-null  object        
 2   vendor_postal_code                  264064 non-null   object        
 3   contract_date                       1021875 non-null  datetime64[ns]
 4   description_en                      1020670 non-null  object        
 5   contract_value                      1021875 non-null  float64       
 6   original_value                      755982 non-null   float64       
 7   amendment_value                     636758 non-null   float64       
 8   comments_en                         515245 non-null   object        
 9   commodity_type                      615916 non-null   object        

# Deduplication

Since this data is reported by government there might be occurences where one entry was made teice or thrice which is a human error.

Given the data quality limitations described above,method fir  detecting duplicate entries were used:

* Because the source data is a transactional ledger, true duplicates are separated from amendments by enforcing a strict 5-variable match (Procurement ID, Department, Master Name, Date, Contract Value).


In [144]:

# Keeping rows with essential fields
clean_df = clean_df[
    clean_df['master_name'].notna() &
    clean_df['procurement_id'].notna() &
    clean_df['owner_org'].notna()
].copy()

# Remove rows that are exact duplicates across key identifying fields

clean_df['is_duplicate'] = clean_df.duplicated(
    subset=[
        'procurement_id',
        'master_name',
        'contract_value',
        'contract_date',
        'owner_org'
    ],
    keep='first'
)

# duplicates_df (with duplicate entries)
duplicates_df = clean_df[clean_df['is_duplicate']].copy()

# non_duplicates_df (with non duplicate entries)
non_duplicates_df = clean_df[~clean_df['is_duplicate']].copy()


print("Total rows:", len(clean_df))
print("Duplicate rows:", len(duplicates_df))
print("Duplicate %:", round(len(duplicates_df) / len(clean_df) * 100, 2), "%")


Total rows: 1021728
Duplicate rows: 12945
Duplicate %: 1.27 %

Sample duplicates:
         reference_number procurement_id vendor_postal_code contract_date  \
298  C-2018-2019-Q4-00042        9000075                NaN    2019-01-09   
300  C-2018-2019-Q4-00044        9000076                NaN    2019-01-10   
306  C-2018-2019-Q4-00051        9000082                NaN    2019-01-07   
308  C-2018-2019-Q4-00053        9000084                NaN    2019-02-04   
313  C-2018-2019-Q4-00058        9000088                NaN    2019-02-04   

                                        description_en  contract_value  \
298  Information technology and telecommunications ...        124526.0   
300  Information technology and telecommunications ...         20396.5   
306  Information technology and telecommunications ...        108480.0   
308  Information technology and telecommunications ...         78365.5   
313  Information technology and telecommunications ...        122379.0   

     origi

In [145]:
non_duplicates_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1008783 entries, 0 to 1021874
Data columns (total 24 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   reference_number                    1008782 non-null  object        
 1   procurement_id                      1008783 non-null  object        
 2   vendor_postal_code                  258651 non-null   object        
 3   contract_date                       1008783 non-null  datetime64[ns]
 4   description_en                      1007650 non-null  object        
 5   contract_value                      1008783 non-null  float64       
 6   original_value                      744758 non-null   float64       
 7   amendment_value                     626372 non-null   float64       
 8   comments_en                         505530 non-null   object        
 9   commodity_type                      605886 non-null   object        
 10 

# Detecting and grouping contract with their amendments

The Open Government CSV is a transactional record. This means that if a contract is awarded in 2018 and amended three times over the next three years, it appears in the dataset as four independent rows. Without a method to link these rows, an analysis would incorrectly treat these as four separate contracts, vastly overcounting the number of active projects and failing to calculate the total final cost of a single initiative.

To solve this, the pipeline generates a unique, internal primary key called **contract_id**. Because a **procurement_id** (the government's reference number) is occasionally reused across different departments or fiscal years, a single identifier is insufficient for grouping.

So we group the data across three variables:

1.	**procurement_id**: The primary link between entries.
2.	**start_year**: Ensures that old, closed contracts with reused IDs are not merged with new ones.
3.	**owner_org**: Prevents collisions between different departments

Once rows are grouped, we must distinguish between the "Base Contract" and subsequent "Amendments".

We apply a binary logic based on the amendment_value field:
* Flag = 1: If the amendment_value is not null and is not equal to zero, the row is identified as an amendment.
* Flag = 0: If the value is null or zero, it is treated as an initial award.

This flag is critical for the spending model. It allows us to recognize that a contract’s financial status has changed, triggering a recalculation of the yearly spend rate from that point forward.


In [147]:
import numpy as np
import pandas as pd

df_amend = non_duplicates_df.copy()

# Ensure procurement_id is of a consistent string type to prevent comparison errors during groupby
df_amend['procurement_id'] = df_amend['procurement_id'].astype(str)

# Column for grouping
group_cols = ['procurement_id', 'start_year', 'owner_org']
grouped = df_amend.groupby(group_cols)


random_id_pool = np.random.choice(np.arange(100000, 99999999), size=grouped.ngroups, replace=False)


df_amend['contract_id'] = random_id_pool[grouped.ngroup()]

# Creating is_amendment flag to ensure its a original contract or amendment that occured later

df_amend['amendment_value'] = pd.to_numeric(df_amend['amendment_value'], errors='coerce')

is_amend_condition = df_amend['amendment_value'].notna() & (df_amend['amendment_value'] != 0)

# Convert boolean into 1 and 0
df_amend['is_amendment'] = is_amend_condition.astype(int)


# peek at a few columns to make sure it looks right
check_cols = ['contract_id', 'procurement_id', 'amendment_value', 'is_amendment']
print(df_amend[check_cols].head(10))

   contract_id procurement_id  amendment_value  is_amendment
0     34856465       P2000002         23645.25             1
1     99279124       P2000004         15043.13             1
2     76478661       P2000009             0.00             0
3     28731497       P2000011             0.00             0
4     53898288       P2000014             0.00             0
5     15733536       P2000007         19584.31             1
6      5364223       P2000008              NaN             0
7     12168365       P2100001              NaN             0
8     74832621       P2100005              NaN             0
9     12168365       P2100001         20000.00             1


In [161]:
df_amend.to_csv('amend_df.csv')

In [162]:
df_amend.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1008783 entries, 0 to 1021874
Data columns (total 27 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   reference_number                    1008782 non-null  object        
 1   procurement_id                      1008783 non-null  object        
 2   vendor_postal_code                  258651 non-null   object        
 3   contract_date                       1008783 non-null  datetime64[ns]
 4   description_en                      1007650 non-null  object        
 5   contract_value                      1008783 non-null  float64       
 6   original_value                      744758 non-null   float64       
 7   amendment_value                     626372 non-null   float64       
 8   comments_en                         505530 non-null   object        
 9   commodity_type                      605886 non-null   object        
 10 

Here we create amend_year to see when did the amendment occured for that contract which will be useful to us in next steps when we are creating yealy spread calcualtion

In [158]:
import pandas as pd
import numpy as np

# Get years from clean_reporting_period
extracted_yy = df_amend['clean_reporting_period'].astype(str).str[4:6]

full_amend_year = pd.to_numeric('20' + extracted_yy, errors='coerce').astype('Int64')

# Condition to apply the amend_year
df_amend['amend_year'] = np.where(df_amend['is_amendment'] == 1, full_amend_year, pd.NA)

# peek
print(df_amend[['clean_reporting_period', 'is_amendment', 'amend_year']].head(10))

  clean_reporting_period  is_amendment amend_year
0              201920-Q4             1       2020
1              201920-Q4             1       2020
2              201920-Q4             0       <NA>
3              201920-Q4             0       <NA>
4              201920-Q4             0       <NA>
5              202021-Q1             1       2021
6              202021-Q1             0       <NA>
7              202021-Q1             0       <NA>
8              202021-Q2             0       <NA>
9              202021-Q3             1       2021


# Normalizing by Year (The Linear Spread)

# The Spreading Logic

The spread_yearly_value_fast function is the engine that converts a single contract entry into a multi-year timeline. It operates on the assumption of Linear Spending: that a department utilizes its contracted funds at an even rate across every active year of the project. While actual spending might be "lumpy" (e.g., front-loaded for equipment or back-loaded for final deliverables), the linear model provides the most accurate estimation possible given that year-by-year invoice data is not public.






**The  Rate Calculation**

The algorithm does not simply divide the final contract value by the total number of years. Instead, it processes each contract through a series of "segments" to account for amendments that change the contract’s value or duration over time.
For every new entry (the original award or a subsequent amendment) within a contract’s lifecycle, the function recalculates the annual spend rate as follows:

\$$\text{Yearly Rate} = \frac{\text{Total Value of Effective Entry} - \text{Cumulative Amount Already Spent}}{\text{Years Remaining in Effective Timeline}}$$

To understand how the algorithm handles overlapping information from amendments, consider a single contract group with four distinct entries published at different times:
* Entry #1 (Initial): Published: 2005. Timeline: 2005–2009. Value: $10M.

* Entry #2 (Amendment): Published: 2007. Timeline: 2005–2011. Value: $10M.

* Entry #3 (Amendment): Published: 2010. Timeline: 2005–2011. Value: $15M.

* Entry #4 (Amendment): Published: 2011. Timeline: 2005–2015. Value: $20M.

The algorithm determines the Effective Entry for each year by selecting the most recent data available at that specific point in time:


In [166]:


# Filter the dataset to focus strictly on contracts starting between 2017 and 2026.
# This scopes the analysis to recent data and significantly reduces compute time.
by_year_df = df_amend.copy()
by_year_df = by_year_df[(by_year_df['start_year'] >= 2017) & (by_year_df['start_year'] <= 2026)].copy()




def spread_yearly_value_fast(contract_id, group):
    """
    Spreads the total contract value across the years it spans.
    Handling amendments by recalculating the remaining value and spreading it
    over the remaining years
    """
    group = group.sort_values('contract_date').copy()

    # Data safety check: Ensure the end year never precedes the start year
    group.loc[group['start_year'] > group['effective_end_year'], 'effective_end_year'] = group['start_year']

    # We use a dictionary to represent the contract's lifespan.
    timeline = {}
    is_first_entry = True

    for idx, row in group.iterrows():
        # Skip rows with critically missing timeline data
        if pd.isna(row['contract_date']) or pd.isna(row['start_year']) or pd.isna(row['effective_end_year']):
            continue

        pub_year = int(row['contract_date'].year)
        start_y = int(row['start_year'])
        end_y = int(row['effective_end_year'])

        # Base contracts start spreading from their actual start_year.
        # Amendments start spreading from the year they were published/signed.
        effective_start = start_y if is_first_entry else pub_year
        amend_y = row.get('amend_year', pd.NA)

        for y in range(effective_start, end_y + 1):
            existing_flag = timeline.get(y, {}).get('is_amendment', 0)

            # Determine if the current row triggers an amendment flag for this year
            amend_flag = 0
            if row['is_amendment'] == 1 and pd.notna(amend_y):
                if y >= amend_y:
                    amend_flag = 1
            elif row['is_amendment'] == 1 and pd.isna(amend_y):
                amend_flag = 1
            # prevent any future rows from downgrading it back to a base contract (0).
            final_flag = max(existing_flag, amend_flag)

            # Record the state of the contract for this specific year
            timeline[y] = {
                'contract_value': row['contract_value'],
                'entry_start': effective_start,
                'entry_end': end_y,
                'entry_id': idx,
                'is_amendment': final_flag
            }
        is_first_entry = False

    if not timeline:
        return []

    # Calculate financial thread
    sorted_years = sorted(timeline.keys())
    cumulative_spent = 0
    yearly_values = []
    processed_entries = set()
    current_per_year_rate = 0

    for current_year in sorted_years:
        entry = timeline[current_year]
        entry_id = entry['entry_id']
        original_row = group.loc[entry_id]


        if entry_id not in processed_entries:
            remaining_value = entry['contract_value'] - cumulative_spent
            years_to_spread = (entry['entry_end'] - entry['entry_start']) + 1

            # Find the new annualized run rate
            if years_to_spread > 0:
                current_per_year_rate = remaining_value / years_to_spread
            else:
                current_per_year_rate = remaining_value

            processed_entries.add(entry_id)

        # Build the final row representing this single year
        yearly_values.append({
            'reference_number': original_row['reference_number'],
            'procurement_id': original_row['procurement_id'],
            'vendor_postal_code': original_row['vendor_postal_code'],
            'contract_date': original_row['contract_date'],
            'year': current_year,
            'yearly_value': current_per_year_rate,
            'original_value': original_row['original_value'],
            'commodity_type': original_row['commodity_type'],
            'country_of_vendor': original_row['country_of_vendor'],
            'solicitation_procedure': original_row['solicitation_procedure'],
            'indigenous_business_excluding_psib': original_row['indigenous_business_excluding_psib'],
            'owner_org': original_row['owner_org'],
            'owner_org_title': original_row['owner_org_title'],
            'master_name': original_row['master_name'],
            'clean_reporting_period': original_row['clean_reporting_period'],
            'effective_start_year': original_row['effective_start_year'],
            'effective_end_year': original_row['effective_end_year'],
            'contract_id': contract_id,
            'is_amendment': entry['is_amendment']
        })

        # Track total money distributed
        cumulative_spent += current_per_year_rate

    return yearly_values


print("Calculating yearly spread...")
start_time = time.time()


all_expanded_rows = []

for c_id, group in by_year_df.groupby('contract_id'):
    all_expanded_rows.extend(spread_yearly_value_fast(c_id, group))

# Construct the final annualized dataset
yearly_df = pd.DataFrame(all_expanded_rows)

end_time = time.time()
print(f"Done! Finished in {round(end_time - start_time, 2)} seconds.")

# Peek
print(yearly_df.head(15))

Dataset filtered! Total unique contracts to process: 479584
Calculating yearly spread...
Done! Finished in 827.54 seconds.
        reference_number              procurement_id vendor_postal_code  \
0   C-2018-2019-Q1-04904  W8482-182556/001/D Mar P 5                NaN   
1   C-2024-2025-Q3-00840                   RAS243279                K2C   
2   C-2024-2025-Q3-00840                   RAS243279                K2C   
3   C-2020-2021-Q2-02401                  4502111382                NaN   
4   C-2018-2019-Q1-00303            21807-18-2610671                NaN   
5   C-2018-2019-Q1-00303            21807-18-2610671                NaN   
6   C-2018-2019-Q1-00303            21807-18-2610671                NaN   
7   C-2018-2019-Q4-00180            21807-18-2610671                NaN   
8   C-2024-2025-Q4-00925                   RAS243186                K2C   
9   C-2020-2021-Q2-00225                   20210099A                NaN   
10  C-2020-2021-Q2-00225                   20210099A

In [167]:
yearly_df

,reference_number,procurement_id,vendor_postal_code,contract_date,year,yearly_value,original_value,commodity_type,country_of_vendor,solicitation_procedure,indigenous_business_excluding_psib,owner_org,owner_org_title,master_name,clean_reporting_period,effective_start_year,effective_end_year,contract_id,is_amendment
0,C-2018-2019-Q1-04904,W8482-182556/001/D Mar P 5,NaN,2018-05-09,2018,311499.520000,311499.52,S,CA,OB,N,dnd-mdn,National Defence,KAYCOM,201819-Q1,2018,2018,100145,0
1,C-2024-2025-Q3-00840,RAS243279,K2C,2024-12-17,2024,815796.440000,1631592.88,G,CA,OB,N,nrc-cnrc,National Research Council Canada,NOVA NETWORKS,202425-Q3,2024,2025,100381,0
2,C-2024-2025-Q3-00840,RAS243279,K2C,2024-12-17,2025,815796.440000,1631592.88,G,CA,OB,N,nrc-cnrc,National Research Council Canada,NOVA NETWORKS,202425-Q3,2024,2025,100381,0
3,C-2020-2021-Q2-02401,4502111382,NaN,2020-06-24,2020,13054.390000,13054.39,C,CA,TC,N,dnd-mdn,National Defence,CANDOR BUILD,202021-Q2,2020,2020,100613,0
4,C-2018-2019-Q1-00303,21807-18-2610671,NaN,2017-05-19,2017,53160.000000,74500.00,S,CA,OB,N,csc-scc,Correctional Service of Canada,DR ALI SALARI,201819-Q1,2017,2019,100893,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
776550,C-2025-2026-Q3-00381,3000807244,K1G,2025-11-05,2027,3832.500000,11497.50,S,CA,TN,N,ec,Environment and Climate Change Canada,CONSERVATION METRICS,202526-Q3,2025,2027,99999467,0
776551,C-2022-2023-Q1-00041,3000737637,M5S,2021-12-13,2021,82666.666667,104000.00,S,CA,TN,N,nrcan-rncan,Natural Resources Canada,JACK SATTERLY GEOCHRONOLOGY LAB,202223-Q1,2021,2023,99999477,0
776552,C-2022-2023-Q1-00041,3000737637,M5S,2021-12-13,2022,82666.666667,104000.00,S,CA,TN,N,nrcan-rncan,Natural Resources Canada,JACK SATTERLY GEOCHRONOLOGY LAB,202223-Q1,2021,2023,99999477,1
776553,C-2022-2023-Q1-00041,3000737637,M5S,2021-12-13,2023,82666.666667,104000.00,S,CA,TN,N,nrcan-rncan,Natural Resources Canada,JACK SATTERLY GEOCHRONOLOGY LAB,202223-Q1,2021,2023,99999477,1


In [168]:
yearly_df.to_csv('yearly_df.csv')

In [155]:
display(clean_df.loc[clean_df['procurement_id'] == 'W8482-182556/001/D Mar P 5'])

,reference_number,procurement_id,vendor_postal_code,contract_date,description_en,contract_value,original_value,amendment_value,comments_en,commodity_type,...,owner_org,owner_org_title,contract_year,start_year,end_year,master_name,clean_reporting_period,effective_start_year,effective_end_year,is_duplicate
578095,C-2018-2019-Q1-04904,W8482-182556/001/D Mar P 5,NaN,2018-05-09,"Measuring, controlling, laboratory, medical an...",311499.52,311499.52,0.0,THIS CONTRACT WAS COMPETITIVELY SOURCED.,S,...,dnd-mdn,National Defence,2018,2018.0,2018.0,KAYCOM,201819-Q1,2018,2018,False
